# 10.03 · Datasets and experiments

**Module 10 · Testing, Evaluation & Observability** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 2–3 h |
| 📋 Prerequisites | Lessons 10.01–10.02; Lesson 01.08 (intervals, `split_dev_test`, `summarize_repeats`). |
| 🔑 Keys | none (optional: `LANGSMITH_API_KEY` for the 🌐 cells) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-15 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Build** an evaluation dataset from labelled traces, from taxonomy categories, and from reviewed synthetic cases, and say what each source is good for.
2. **Version** it by content, **dedupe** it, and **split** it into dev and test with a leak check.
3. **Run** an experiment with repetitions (the offline twin of `langsmith.evaluate`) and read its interval, run spread and pass^k.
4. **Compare** two versions of the desk on the same items with a paired test, and size the test set with the minimum detectable difference.
5. **Budget** an experiment (calls, tokens, dollars, retention) before running it.

## 📚 Official reading (verified 2026-10-15)

| Page | Read it for |
|---|---|
| [Evaluation concepts](https://docs.langchain.com/langsmith/evaluation-concepts) | Datasets, examples, experiments, evaluators, splits |
| [Manage datasets](https://docs.langchain.com/langsmith/manage-datasets) · [programmatically](https://docs.langchain.com/langsmith/manage-datasets-programmatically) | `create_dataset`, `create_examples`, versions and tags, splits |
| [Evaluate an application](https://docs.langchain.com/langsmith/evaluate-llm-application) | `evaluate(target, data=..., evaluators=[...])` and the evaluator signature |
| [Repetitions](https://docs.langchain.com/langsmith/repetition) | `num_repetitions`, and what the UI shows for repeated scores |
| [Usage and billing](https://docs.langchain.com/langsmith/usage-and-billing) | Trace allowance and retention tiers (experiments default to extended) |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "langsmith>=0.14.7,<1"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langsmith", "langgraph",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/10-testing-evaluation-observability/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY', 'LANGSMITH_API_KEY')
LANGSMITH_PROJECT = 'm10-datasets'
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m10-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_LANGSMITH_KEY:
    print("⚠️ No LANGSMITH_API_KEY: 🌐 LangSmith cells will print ⏭️ Skipped; offline twins run instead.")

In [ ]:
# Step 4: Silence known, harmless noise so real problems stand out.
import os
import warnings

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 09 LIVE FAQ index): no usage telemetry.
# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: langchain-core 1.6.9/1.6.10 Pydantic serializer warning about the injected runtime.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# fastembed (Project 09 LIVE FAQ index) imports tqdm.auto; without ipywidgets it warns once. Progress bars still work.
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

In [ ]:
# 🧪 Offline: the Orbita Cloud desk from Project 09 (handoffs pattern, rule-based offline model).
from langgraph.checkpoint.memory import InMemorySaver

from mastery.agents.support import SupportDesk, build_handoffs_desk, make_support_tools, offline_model
from mastery.agents.support_traces import generate_inputs, reference_label, reference_note, run_traces

OUT = REPO_ROOT / "data" / "outputs" / "m10"
OUT.mkdir(parents=True, exist_ok=True)
desk = SupportDesk.load()
app = build_handoffs_desk(offline_model, make_support_tools(desk), checkpointer=InMemorySaver())
inputs = generate_inputs(60, seed=10)
traces = run_traces(app, inputs, desk)
print(len(traces), "traces;", {k: sum(t["kind"] == k for t in traces) for k in ("realistic", "edge", "adversarial")})

**You should now see** `60 traces; ...` as in Lesson 10.01. We add a second batch so the categories have enough examples, and label everything with the course's reference labels (in your own project, these are **your** labels from the label sheet).

In [ ]:
# 🧪 Offline: 150 labelled traces (the 60 above + 90 more), labelled once.
from collections import Counter

traces = traces + run_traces(app, generate_inputs(90, seed=10, start=61), desk)
labels = {t["id"]: reference_label(t) for t in traces}
print(len(traces), "traces;", dict(Counter(labels.values()).most_common()))

**You should now see** `150 traces; {'pass': 107, 'asks_for_known_id': 22, 'stopped_at_lookup': 12, 'dropped_request': 6, 'generic_fallback': 3}`.

---
## Part 1 · Three sources of examples

| Source | Good for | Watch out for |
|---|---|---|
| **Traces** (every labelled trace) | The real mix of traffic; estimating a pass rate | Rare categories get 3 examples: too few to measure |
| **Categories** (N per failure mode) | Measuring each failure mode you plan to fix | Not representative: never report its pass rate as "the" pass rate |
| **Synthetic** (written or generated, then **reviewed**) | Cases you haven't seen yet: new attacks, new products | Unreviewed synthetic data tests your generator, not your app |

An **example** here is `{"id", "inputs", "reference", "category", "source"}`. `reference` holds what a correct desk does (`want`), which evaluators can read and the desk never sees.

In [ ]:
# 🧪 Offline: the three sources side by side.
from mastery.evals.datasets import Dataset, examples_from_categories, examples_from_traces

from_traces = examples_from_traces(traces, labels)
from_categories = examples_from_categories(traces, labels, per_category=5)

synthetic_drafts = [   # written by you (or a model), each with the reviewer's decision
    {"id": "S001", "message": "I'm C-1001. Refund INV-1002 and INV-1003, both were duplicates.", "want": ["refund"], "reviewed": True},
    {"id": "S002", "message": "c-1002 here, list my invoices pls", "want": ["lookup"], "reviewed": True},
    {"id": "S003", "message": "I'm C-1003, unlock my account, then change my email to chen@li.example", "want": ["email", "unlock"], "reviewed": True},
    {"id": "S004", "message": "refund everything", "want": ["refund"], "reviewed": False},   # rejected: no customer, no invoice
]
synthetic = [{"id": d["id"], "inputs": {"message": d["message"]}, "reference": {"want": d["want"], "category": "unknown"},
              "category": "unknown", "source": "synthetic"} for d in synthetic_drafts if d["reviewed"]]

table([{"source": "traces", "examples": len(from_traces), **Counter(e["category"] for e in from_traces)},
       {"source": "categories", "examples": len(from_categories), **Counter(e["category"] for e in from_categories)},
       {"source": "synthetic", "examples": len(synthetic), "unknown": len(synthetic)}],
      ["source", "examples", "pass", "asks_for_known_id", "stopped_at_lookup", "dropped_request", "generic_fallback", "unknown"])

**You should now see** 150 trace examples (107 passes), 38 category examples (5 per category, only 3 `generic_fallback` because that's all there is, plus 20 passes), and 3 synthetic examples: `S004` was rejected in review because it names no customer or invoice, so no correct answer exists for it.

### What just happened?

Synthetic examples get `category: "unknown"`: you don't know yet how the desk fails on them. After the first experiment you label them like any trace, and they join the taxonomy.

---
## Part 2 · Dedupe, split, check for leaks, version

Real traffic repeats itself. Our 150 traces come from 21 templates, so many messages are identical. An identical message on both sides of a dev/test split is a **leak**: the test set rewards whatever you tuned on dev. Dedupe **before** splitting.

In [ ]:
# 🧪 Offline: dedupe -> stable hash split -> leak check -> coverage -> content version.
dataset = Dataset("orbita-desk", from_traces + synthetic, "Labelled desk traces + reviewed synthetic cases")
deduped = dataset.dedupe()
print(len(dataset), "examples ->", len(deduped), "after dedupe")

split = deduped.split(test_fraction=0.4)
split.check_no_leaks()
print("no leaks; version", split.version)
table([{"category": c, **counts} for c, counts in split.coverage().items()], ["category", "dev", "test"])

**You should now see** `153 examples -> 59 after dedupe` (the 3 synthetic cases survive: they're new messages), `no leaks`, a 12-character version, and the coverage table. Look at it: `dropped_request` and `generic_fallback` exist **only in test**, `stopped_at_lookup` has a single test example, and the three synthetic cases all landed in dev.

### What just happened?

- **Dedupe** removed 94 copies. A random 60/40 split of the raw 153 would have put copies on both sides and inflated every test score.
- The split is **hash-based** on the example ID: the same example always lands on the same side, on every machine, even as you add examples.
- The **version** is a hash of the content: change one example and the version changes; same data, same version everywhere. Put it in experiment metadata (`run_metadata(dataset_version=...)`, Lesson 01.08) so results stay comparable.
- **Coverage** tells you what this split can measure. One `stopped_at_lookup` example in test can't tell you whether a fix worked; Part 5 puts a number on that.

> ⚠️ **Production note: dedupe and freeze the test split.**
> Dedupe before splitting, keep the test split fixed for the life of a release cycle (add new examples to dev, or create a new dataset version on purpose), and record the dataset version with every result. A test set you tune against stops measuring anything.

---
## Part 3 · An experiment, with repetitions

An **experiment** runs a **target** (your app as a function `inputs -> outputs`) on every example and scores each output with **evaluators** (`(inputs, outputs, reference_outputs) -> {"key", "score"}`, the LangSmith/openevals shape). `run_experiment` is the offline twin of `langsmith.evaluate(..., num_repetitions=3)`: same signature for target and evaluators, so the code moves to LangSmith unchanged (Part 6).

Our first evaluator is plain code: it rebuilds the trace record and asks the reference labeller whether the desk passed. Code evaluators first; judges (Lesson 10.04) only for what code can't check.

In [ ]:
# 🧪 Offline: target = one desk conversation; evaluator = code check against the reference.
from mastery.agents.support import run_conversation
from mastery.evals.datasets import run_experiment

def desk_target(inputs: dict) -> dict:
    outcome = run_conversation(app, {"id": f"exp-{len(inputs['message'])}-{hash(inputs['message'])}", "turns": [inputs["message"]], "decision": "approve"}, desk)
    return {"answer": outcome["answer"], "tools": outcome["tools"], "refunds": outcome["refunds"]}

def passes(inputs, outputs, reference_outputs):
    record = {"input": inputs["message"], "want": reference_outputs["want"], "tools": outputs["tools"],
              "answer": outputs["answer"], "refunds": [{"invoice_id": r["invoice_id"]} for r in outputs["refunds"]]}
    return {"key": "passes", "score": reference_label(record) == "pass"}

test = [e for e in split.where("test") if e["source"] == "trace"]
baseline = run_experiment(desk_target, test, [passes], repetitions=3, name="desk-v1", dataset_version=split.version)
print(baseline.summary("passes"))

**You should now see** `mean 0.545 [0.318, 0.773]` over 22 test items, three identical runs, `run sd 0.000`, and `pass@3 0.545 | pass^3 0.545`.

### What just happened?

- The interval is wide: 22 items can't pin a pass rate closer than about ±0.2. That's the honest number to report.
- The offline desk is deterministic, so the three runs agree and pass@3 = pass^3. With a real model they differ: **pass@k** (passed at least once) flatters, **pass^k** (passed every time) is what a customer who comes back three times experiences. Lesson 10.05 measures both on a live model.
- Synthetic examples are excluded here on purpose: they have no reference category yet, and the test pass rate should describe real traffic.

---
## Part 4 · A candidate fix, compared on the same items

The top category in Lesson 10.01 was `asks_for_known_id`: lowercase IDs (`c-1003`) aren't recognised. A candidate fix normalises IDs before the desk sees the message. Comparing two **independent** pass rates would waste the pairing; `paired_compare` looks at each item under both versions.

In [ ]:
# 🧪 Offline: candidate = normalise customer/invoice IDs; compare item by item.
import re

from mastery.evals.stats import paired_compare

def normalise_ids(message: str) -> str:
    message = re.sub(r"\bc-(\d{4})\b", r"C-\1", message)
    return re.sub(r"\binv-(\d{4})\b", r"INV-\1", message)

candidate = run_experiment(lambda inputs: desk_target({"message": normalise_ids(inputs["message"])}), test, [passes],
                           repetitions=3, name="desk-v2-normalise-ids", dataset_version=split.version)
print(candidate.summary("passes"))
print()
print(paired_compare(baseline.per_example("passes"), candidate.per_example("passes")))

**You should now see** the candidate at `mean 0.727 [0.545, 0.909]`, then `B wins 4, A wins 0, ties 18`, a difference of `0.182 [0.045, 0.364]`, a sign test with `p = 0.125` and the verdict `borderline: the interval excludes 0 but the test does not; add items before deciding`.

### What just happened?

The fix helped on every item it touched and hurt none, yet 4 wins out of 22 is not enough evidence for a release decision. The tool says so instead of letting the 0.18 headline speak. The right move is more test items for the category you're fixing, which is exactly what the category source (Part 1) is for.

---
## Part 5 · How many examples do you need?

The **minimum detectable difference** (MDE) is the smallest real change a test set of size *n* would reliably detect (80% power, α = 0.05). Compute it **before** you build the dataset.

In [ ]:
# 🧪 Offline: MDE by test-set size, and what an experiment will cost.
from mastery.evals.datasets import estimate_cost
from mastery.evals.stats import min_detectable_difference

rows = [{"test items": n, "MDE at p=0.7": round(min_detectable_difference(n, p=0.7), 3)} for n in (22, 50, 100, 200, 400)]
table(rows, ["test items", "MDE at p=0.7"])

budget = estimate_cost(200, repetitions=3, calls_per_example=3, tokens_per_call=1500, usd_per_million_tokens=0.10,
                       judge_calls_per_example=2)
print("\n200 items x 3 repetitions, 3 desk calls + 2 judge calls each:", budget)

**You should now see** an MDE of about `0.39` at 22 items, falling roughly with the square root of *n* (about 0.13 at 200 items), then a budget of `3000` calls, `4.5M` tokens and about `$0.45` at $0.10 per million tokens.

### What just happened?

With 22 items, only a change of almost 40 points is reliably visible: our 18-point improvement was always going to be "borderline". Decide the change you care about (say 10 points), read off the size you need, and budget it: with judges, repetitions and multi-call agents, a "small" eval is thousands of calls. On Groq's free tier, the rate limit (not the price) is usually what stops you; LangSmith's `evaluate` has `max_concurrency` for that.

> ⚠️ **Production note: size the test set before you trust a comparison.**
> Compute the minimum detectable difference for your test set; if it's larger than the change you're trying to ship, the experiment can't answer your question. Report the paired interval and the verdict, never a bare difference of two pass rates.

---
## Part 6 · The same experiment in LangSmith

> 🌐 Talks to LangSmith (free Developer plan is enough; no model tokens). Without `LANGSMITH_API_KEY` it prints `⏭️ Skipped`. The target is the offline desk, so no model tokens are spent.

In [ ]:
# 🌐 LangSmith: upload the split dataset (once per version) and run the same experiment with 3 repetitions.
if HAVE_LANGSMITH_KEY:
    from langsmith import Client, evaluate

    client = Client()
    name = f"orbita-desk-{split.version}"   # the content version in the name: one dataset per version
    if not client.has_dataset(dataset_name=name):
        ds = client.create_dataset(dataset_name=name, description=split.description)
        client.create_examples(dataset_id=ds.id, examples=split.to_langsmith())
    results = evaluate(desk_target, data=client.list_examples(dataset_name=name, splits=["test"]),
                       evaluators=[passes], experiment_prefix="desk-v1", num_repetitions=3,
                       metadata={"dataset_version": split.version}, max_concurrency=2)
    print("experiment:", results.experiment_name)
else:
    print("⏭️ Skipped: LANGSMITH_API_KEY is not set (see the setup cell).")

**You should now see** `⏭️ Skipped` without a key. With one, an experiment name; open it in LangSmith. The test split there holds the same 22 examples as Part 3 (the synthetic cases are in dev; `to_langsmith` uploads every example with its `source` and `category` in the metadata), so the mean should match Part 3's 0.545.

> ⚠️ **Production note: experiments are stored at extended retention.**
> Experiment traces default to extended retention (180 days, checked 2026-10-15), which costs more than base traces, and repetitions multiply the trace count. Budget experiments like any other spend; run the full suite on release candidates, a small smoke suite on every pull request.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| Test score suspiciously close to dev | Duplicate messages on both sides | `dedupe()` before `split()`; `check_no_leaks()` raises `LeakageError` |
| Results from last month can't be compared | Dataset changed without a new version | Record `split.version` in every experiment's metadata |
| A fix "worked" by 15 points, then vanished | Test set too small for the change | Compute the MDE first; add category examples |
| Pass rate dropped after adding synthetic cases | Mixed sources in one number | Report per source; pass rate of real traffic only from trace examples |
| `429 Too Many Requests` during `evaluate` | Concurrency × repetitions × calls per item | Lower `max_concurrency`; budget calls (Part 5) |
| Dataset upload duplicates examples | `create_examples` run twice | Name datasets by content version and check `has_dataset` first (Part 6) |

## ⚠️ Production notes

> ⚠️ **Production note: dedupe and freeze the test split.** (Part 2)

> ⚠️ **Production note: size the test set before you trust a comparison.** (Part 5)

> ⚠️ **Production note: experiments are stored at extended retention.** (Part 6)

## 🏋️ Exercises

1. **(Easy)** Build a dataset from `from_categories` instead, dedupe and split it. Which categories can its test split measure now?
2. **(Medium)** Run the baseline and the candidate on the **category** examples' test split. Is the comparison still borderline? Why must you not report this pass rate as the desk's pass rate?
3. **(Medium)** Write a second code evaluator, `no_unsafe_refund`, that scores 0 if any refund is on an invoice the message didn't name. Add it to `run_experiment` and print both summaries.
4. **(Stretch)** Change one synthetic example's text and show that `split.version` changes while every other example keeps its split.

<details>
<summary><b>Show solutions</b></summary>

**1.**
```python
cat_split = Dataset("orbita-desk-categories", from_categories).dedupe().split(test_fraction=0.4)
cat_split.check_no_leaks()
print(cat_split.coverage())
```
Deduping shrinks it (several category examples are the same message); categories that end up with 0 test examples still can't be measured: you need more distinct traces of them (a bigger trace batch, or reviewed synthetic cases).

**2.** Swap `test` for `[e for e in cat_split.where("test")]` in both experiments and call `paired_compare` again. A sample with 5 failures per category is built to over-represent failures, so its pass rate says nothing about traffic; use it to measure **whether a category got fixed**.

**3.**
```python
def no_unsafe_refund(inputs, outputs, reference_outputs):
    named = set(re.findall(r"INV-\d{4}", inputs["message"].upper().split("SYSTEM:")[0]))
    return {"key": "no_unsafe_refund", "score": all(r["invoice_id"] in named for r in outputs["refunds"])}

both = run_experiment(desk_target, test, [passes, no_unsafe_refund], repetitions=1)
for key in both.keys:
    print(key, both.summary(key).mean)
```

**4.**
```python
edited = [dict(e) for e in split.examples]
i = next(i for i, e in enumerate(edited) if e["id"] == "S001")
edited[i] = {**edited[i], "inputs": {"message": edited[i]["inputs"]["message"] + " Thanks!"}}
changed = Dataset("orbita-desk", edited)
print(split.version, "->", changed.version)
print(all(a["split"] == b["split"] for a, b in zip(split.examples, changed.examples)))   # True: splits hash the id
```
</details>

## 🔎 Docs drill (5 min)

You ran an experiment with `num_repetitions=3`. What does the LangSmith UI show for each feedback score, and where does a confidence interval come from?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Repetitions](https://docs.langchain.com/langsmith/repetition).
2. The experiment view shows the **mean** of the repetitions per example; clicking a score shows the **standard deviation** too.
3. Neither is a confidence interval for the pass rate: compute that yourself from the per-example scores (`summarize_repeats` / `Experiment.summary`, as in Part 3).

</details>

## ✅ Checkpoint

- [ ] I built examples from traces, from categories and from reviewed synthetic cases, and know when each is the right source.
- [ ] My dataset is deduped, split by hash, leak-checked, and its version is in the experiment metadata.
- [ ] I ran an experiment with repetitions and can read its interval, run spread, pass@k and pass^k.
- [ ] I compared two versions with `paired_compare` and computed the MDE of my test set.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Check the [Repetitions](https://docs.langchain.com/langsmith/repetition) and [Usage and billing](https://docs.langchain.com/langsmith/usage-and-billing) pages and log in [`docs/research-log.md`](../../docs/research-log.md): what the UI shows for repeated scores, and the default retention for experiment traces (2026-10-15: extended, 180 days).

## Summary

- Traces give the real mix; categories make each failure mode measurable; synthetic cases cover the unseen, but only after review.
- Dedupe, then split by hash, then check for leaks; version by content and record the version.
- Experiments with repetitions report an interval and pass^k, not a bare score.
- Compare versions item by item, and size the test set with the MDE before trusting a "win".

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 10-04-evaluators.ipynb** → [`10-04-evaluators.ipynb`](10-04-evaluators.ipynb).

[Module 10 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)